# YOLO11-seg: water / pillar — train v2 (debug dataset + cấu hình cho dataset nhỏ)

In [ ]:
# 1. Config
from pathlib import Path
import re
import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO

DATASET_REL = "resources/data/yolo_train/total"


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / DATASET_REL).is_dir():
            return p
    raise FileNotFoundError(f"Không tìm thấy {DATASET_REL} từ {start}")


PROJECT_DIR = find_root(Path.cwd())
DATASET_DIR = PROJECT_DIR / DATASET_REL
RUNS_DIR = PROJECT_DIR / "runs"
RUN_NAME = "water_seg_v2"
RUN_DIR = RUNS_DIR / RUN_NAME

CLASS_NAMES = ["pillar", "water"]                    # id0 = trụ, id1 = nước (đúng với label hiện có)
COLORS_BGR  = {0: (0, 0, 255), 1: (255, 140, 0)}     # pillar đỏ, water xanh dương
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
IMGSZ = 640

HAS_GPU = torch.cuda.is_available()
DEVICE = 0 if HAS_GPU else "cpu"
# GPU: dùng bản s. CPU: giữ n. Epoch giảm trên CPU (~16s/epoch ở i7-8650U).
EPOCHS = 200 if HAS_GPU else 120
MODEL_NAME = "yolo11s-seg.pt" if HAS_GPU else "yolo11n-seg.pt"
print("Device:", DEVICE, "| Model:", MODEL_NAME, "| Root:", PROJECT_DIR)

DATA_YAML = DATASET_DIR / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        {
            "path": str(DATASET_DIR.resolve()),
            "train": "images/train",
            "val": "images/val",
            "names": {i: n for i, n in enumerate(CLASS_NAMES)},
        },
        f,
        sort_keys=False,
        allow_unicode=True,
    )

In [ ]:
# 2. Helpers: đọc ảnh / polygon / vẽ annotation
def list_images(split: str) -> list[Path]:
    d = DATASET_DIR / "images" / split
    return sorted(p for p in d.rglob("*") if p.suffix.lower() in IMG_EXT)


def label_path(img_path: Path, split: str) -> Path:
    rel = img_path.relative_to(DATASET_DIR / "images" / split)
    return (DATASET_DIR / "labels" / split / rel).with_suffix(".txt")


def imread(p: Path) -> np.ndarray:
    # imdecode an toàn với đường dẫn Windows/unicode
    return cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)


def read_polygons(lp: Path, w: int, h: int) -> list[tuple[int, np.ndarray]]:
    polys = []
    if not lp.is_file():
        return polys
    for line in lp.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        xy = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        xy[:, 0] *= w
        xy[:, 1] *= h
        polys.append((int(parts[0]), xy.round().astype(np.int32)))
    return polys


def draw_gt(img: np.ndarray, polys, alpha: float = 0.4) -> np.ndarray:
    overlay = img.copy()
    for cls, pts in polys:
        cv2.fillPoly(overlay, [pts], COLORS_BGR[cls])
    out = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    for cls, pts in polys:
        cv2.polylines(out, [pts], True, COLORS_BGR[cls], 2)
        x, y = pts.min(axis=0)
        cv2.putText(out, CLASS_NAMES[cls], (int(x), max(int(y) - 6, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS_BGR[cls], 2)
    return out

In [ ]:
# 3. Thống kê dataset + kiểm tra leakage theo frame
rows = []
for split in ("train", "val"):
    for p in list_images(split):
        img = imread(p)
        h, w = img.shape[:2]
        for cls, pts in read_polygons(label_path(p, split), w, h):
            rows.append(dict(split=split, file=p.name, cls=CLASS_NAMES[cls],
                             n_pts=len(pts), area=cv2.contourArea(pts) / (w * h)))
df = pd.DataFrame(rows)
print(df.groupby(["split", "cls"]).agg(
    instances=("file", "size"), images=("file", "nunique"),
    n_pts_med=("n_pts", "median"), area_med=("area", "median"), area_min=("area", "min"),
).round(4))
print({s: len(list_images(s)) for s in ("train", "val")}, "ảnh")


def frame_id(p: Path):
    m = re.findall(r"\d+", p.stem)
    return int(m[-1]) if m else None


train_ids = [i for i in map(frame_id, list_images("train")) if i is not None]
for p in list_images("val"):
    f = frame_id(p)
    if f is not None and train_ids:
        gap = min(abs(f - t) for t in train_ids)
        flag = "  (cùng cảnh/frame gần train: bình thường với demo, val chỉ để sanity-check)" if gap < 30 else ""
        print(f"{p.name}: cách frame train gần nhất {gap}{flag}")

In [ ]:
# 4. Debug: imshow annotation ảnh ngẫu nhiên
def show_samples(split: str = "train", n: int = 8, cols: int = 4, seed: int = SEED):
    paths = list_images(split)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(paths), size=min(n, len(paths)), replace=False)
    rows_n = int(np.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 3.6 * rows_n))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, idx):
        p = paths[i]
        img = imread(p)
        h, w = img.shape[:2]
        polys = read_polygons(label_path(p, split), w, h)
        ax.imshow(cv2.cvtColor(draw_gt(img, polys), cv2.COLOR_BGR2RGB))
        counts = {CLASS_NAMES[c]: sum(1 for k, _ in polys if k == c) for c in range(len(CLASS_NAMES))}
        ax.set_title(f"{p.name}\n{counts}", fontsize=9)
    fig.suptitle(f"GT annotation — {split}", fontsize=13)
    plt.tight_layout()
    plt.show()


show_samples("train", n=8)
show_samples("val", n=5, cols=5)

In [ ]:
# 5. Train
model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=0,            # tắt early stop: val 5 ảnh quá nhiễu, v1 dừng lúc mới ~66 bước
    imgsz=IMGSZ,
    batch=4,               # 21 ảnh -> ~6 iteration/epoch (v1 chỉ có 3)
    device=DEVICE,
    workers=4 if HAS_GPU else 0,
    cache="ram",
    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=1.0,     # YOLO ép tối thiểu 100 iteration warmup
    # Demo 1 cảnh cố định: aug vừa phải, không cần xoay/biến dạng mạnh
    mosaic=1.0,
    close_mosaic=30,
    copy_paste=0.1,
    mixup=0.0,
    degrees=0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    erasing=0.4,
    # Output
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=True,
    seed=SEED,
    deterministic=True,
)
print("Training completed:", RUN_DIR)

In [ ]:
# 6. Learning curves
res = pd.read_csv(RUN_DIR / "results.csv")
res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for c in ("train/seg_loss", "val/seg_loss", "train/cls_loss", "val/cls_loss"):
    if c in res:
        ax[0].plot(res["epoch"], res[c], label=c)
ax[0].set_title("Loss"); ax[0].legend()
for c in ("metrics/mAP50(M)", "metrics/mAP50-95(M)"):
    if c in res:
        ax[1].plot(res["epoch"], res[c], label=c)
ax[1].set_title("Mask mAP (val)"); ax[1].legend()
for c in ("metrics/precision(M)", "metrics/recall(M)"):
    if c in res:
        ax[2].plot(res["epoch"], res[c], label=c)
ax[2].set_title("Mask P / R (val)"); ax[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7. Validate best.pt (per-class) — chỉ tham khảo, val = 5 ảnh
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))
metrics = best.val(
    data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE, plots=True,
    project=str(RUNS_DIR), name="val_v2", exist_ok=True,
)
print(f"Mask mAP50: {metrics.seg.map50:.3f} | mAP50-95: {metrics.seg.map:.3f}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:8s} P={metrics.seg.p[i]:.3f} R={metrics.seg.r[i]:.3f} "
          f"AP50={metrics.seg.ap50[i]:.3f} AP50-95={metrics.seg.all_ap[i].mean():.3f}")

In [ ]:
# 8. So sánh GT vs Prediction trên val
def compare_gt_pred(split: str = "val", n: int = 6, conf: float = 0.25):
    paths = list_images(split)[:n]
    fig, axes = plt.subplots(len(paths), 2, figsize=(13, 3.8 * len(paths)))
    axes = np.atleast_2d(axes)
    for row, p in zip(axes, paths):
        img = imread(p)
        h, w = img.shape[:2]
        gt = draw_gt(img, read_polygons(label_path(p, split), w, h))
        r = best.predict(img, imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        dets = [f"{CLASS_NAMES[int(c)]}:{s:.2f}"
                for c, s in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())]
        row[0].imshow(cv2.cvtColor(gt, cv2.COLOR_BGR2RGB))
        row[0].set_title(f"GT — {p.name}", fontsize=9)
        row[1].imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        row[1].set_title(f"Pred (conf={conf}) {dets}", fontsize=9)
        for a in row:
            a.axis("off")
    plt.tight_layout()
    plt.show()


compare_gt_pred("val", n=5, conf=0.25)
compare_gt_pred("val", n=5, conf=0.05)  # xem water có bị loại do threshold không